## **Disclaimer:**
As mentioned in the paper the inference is quite expensive, running this locally was impractical for me. For this reason I tryed this workaround, creating a notebook that I would have been able to run on Google Colab, forking the repository inserting this notebook and doing some changes to the paths on the python files to make everything work in Google Colab.

This notebook contains a slightly modified version of uni-ddim-inversion.py and sim.py.

# Required libraries and imports

In [1]:
!git clone https://github.com/MarkBridge11/ZeroFake-Mod.git

Cloning into 'ZeroFake-Mod'...
remote: Enumerating objects: 184, done.
remote: Counting objects: 100% (184/184), done.
remote: Compressing objects: 100% (159/159), done.
remote: Total 184 (delta 89), reused 45 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (184/184), 1.59 MiB | 5.54 MiB/s, done.
Resolving deltas: 100% (89/89), done.


In [2]:
import sys
sys.path.append('/content/ZeroFake-Mod')

In [3]:
!pip install fairscale timm natsort scikit-image einops kornia spacy albumentations pudb invisible-watermark imageio imageio-ffmpeg omegaconf test-tube streamlit nltk fasttext sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.3/266.3 kB 15.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 5.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.7 MB/s eta 0:00:00
  Using cached pybind11-2.13.6-py3-none-any.whl.metadata (9.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.2/89.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1

In [4]:
from functools import partial
from pathlib import Path
from typing import Callable, Dict, List, Optional, Tuple, Union

import numpy as np
import PIL
import torch
import torch.nn.functional as nnf

from diffusers import DDIMScheduler, StableDiffusionPipeline

path = "/home/c01zesh/CISPA-projects/meta_transfer-2023/stable-diffusion/stable-diffusion-v1-4"
path = Path(path).expanduser()
# These variable is not used in my case because here they were importing locally stable-diffusion model. The path should correspond to some path in Linux system

from PIL import Image
from torchvision import transforms

import inspect

import torch
from transformers import CLIPFeatureExtractor, CLIPTextModel, CLIPTokenizer

from diffusers.configuration_utils import FrozenDict
from diffusers.models import AutoencoderKL, UNet2DConditionModel
try:
    from diffusers.pipeline_utils import DiffusionPipeline
except:
    from diffusers.pipelines.pipeline_utils import DiffusionPipeline
from diffusers.pipelines.stable_diffusion import StableDiffusionPipelineOutput
from diffusers.pipelines.stable_diffusion.safety_checker import \
    StableDiffusionSafetyChecker
from diffusers.schedulers import DDIMScheduler,PNDMScheduler, LMSDiscreteScheduler
from diffusers.utils import deprecate, logging, numpy_to_pil
from natsort import natsorted

from PIL import Image
import requests
from torchvision import transforms
from torchvision.transforms.functional import InterpolationMode
from blipmodels import blip_decoder
import spacy

import cv2
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as PSNR
from skimage.metrics import mean_squared_error as MSE

import os
import sys
import argparse

import nltk
from nltk.corpus import brown
from nltk import pos_tag
from nltk.tokenize import word_tokenize
from nltk import download

# Helper functions

In [5]:
# def replace_first_noun(sentence, replacement):
#     doc = nlp(sentence)
#     for token in doc:
#         if token.tag_ in {"NN", "NNS", "NNP", "NNPS"}:
#             return sentence[:token.idx] + replacement + sentence[token.idx + len(token.text):]
#     return sentence
def replace_first_noun(sentence, replacement):
    doc = nlp(sentence)

    first_chunk = None
    for chunk in doc.noun_chunks:
        if first_chunk is None or chunk.start_char < first_chunk.start_char:
            first_chunk = chunk
    # This preliminary part of the function is essential because noun chunks are not sorted by textual order, so it may happen that a noun that is at the end
    # of the phrase is taken

    if first_chunk:
        if first_chunk[0].pos_ == "DET": # If the first token is a determiner (like "the", "a", "an"), preserve it
            noun_start = first_chunk[1].idx # the start is shifted
            noun_end = first_chunk.end_char
            return sentence[:noun_start] + replacement + sentence[noun_end:]
        else:
            return sentence[:first_chunk.start_char] + replacement + sentence[first_chunk.end_char:]

    return sentence

def backward_ddim(x_t, alpha_t: float, alpha_tm1: float, eps_xt):
    """ from noise to image"""
    return (
        alpha_tm1**0.5
        * (
            (alpha_t**-0.5 - alpha_tm1**-0.5) * x_t
            + ((1 / alpha_tm1 - 1) ** 0.5 - (1 / alpha_t - 1) ** 0.5) * eps_xt
        )
        + x_t
    )

def forward_ddim(x_t, alpha_t: float, alpha_tp1: float, eps_xt):
    """ from image to noise, it's the same as backward_ddim"""
    return backward_ddim(x_t, alpha_t, alpha_tp1, eps_xt)

def load_img(path, target_size=512):
    """Load an image, resize and output -1..1"""
    image = Image.open(path).convert("RGB")

    tform = transforms.Compose(
        [
            transforms.Resize((target_size,target_size)), #resizing to 512x512
            transforms.CenterCrop(target_size), #center cropping
            transforms.ToTensor(),
        ]
    )
    image = tform(image)
    return 2.0 * image - 1.0 # converting the image into [-1,1] range

# Redifinition of StableDiffusionPipeline

In [6]:
class StableDiffusionPipeline(DiffusionPipeline):
    def __init__(
        self,
        vae: AutoencoderKL,
        text_encoder: CLIPTextModel,
        tokenizer: CLIPTokenizer,
        unet: UNet2DConditionModel,
        scheduler: Union[DDIMScheduler, PNDMScheduler, LMSDiscreteScheduler],
        #safety_checker: StableDiffusionSafetyChecker = None,
        #feature_extractor: CLIPFeatureExtractor = None
    ):
        super().__init__()

        self.register_modules( # registers the following components in the pipeline
            vae=vae,
            text_encoder=text_encoder,
            tokenizer=tokenizer,
            unet=unet,
            scheduler=scheduler,
        )
        self.forward_diffusion = partial(self.backward_diffusion, reverse_process=True)

    """
    Transform the prompt into text embedding using CLIPTokenizer
    """
    @torch.inference_mode()
    def get_text_embedding(self, prompt):
        text_input_ids = self.tokenizer(
            prompt,
            padding="max_length",
            truncation=True,
            max_length=self.tokenizer.model_max_length,
            return_tensors="pt",
        ).input_ids
        text_embeddings = self.text_encoder(text_input_ids.to(self.device))[0]
        return text_embeddings


    """
    We obtain the latent vector of the input image of the VAE.
    First we retrieve the DiagonalGaussianDistribution (encoding_dist) that it uses it to sample from it or directly from the mean using .mode().
    Then the latents are scaled.
    """
    @torch.inference_mode()
    def get_image_latents(self, image, sample=True, rng_generator=None):
        encoding_dist = self.vae.encode(image).latent_dist # as mentioned in the paper it's a zero-shot approach but we still need to know a VAE and UNet of a LDM
        if sample:
            encoding = encoding_dist.sample(generator=rng_generator)
        else:
            encoding = encoding_dist.mode()
        latents = encoding * 0.18215
        return latents

    """
    Despite the name this function can do both backward and forward DDIM sampling. In our case it is used as inversion and reconstruction exploiting formulas defined above.
    """
    @torch.inference_mode()
    def backward_diffusion(
        self,
        use_old_emb_i=25, # set the inference step from which changes the text embedding to the modified one
        text_embeddings=None,
        old_text_embeddings=None, # not modified prompt
        new_text_embeddings=None, # modified prompt
        latents: Optional[torch.FloatTensor] = None, # initial latent input, x0 if reverse, x_T if reconstruction
        num_inference_steps: int = 50,
        guidance_scale: float = 7.5, # strength of classifier-free guidance
        callback: Optional[Callable[[int, int, torch.FloatTensor], None]] = None,
        callback_steps: Optional[int] = 1,
        reverse_process: True = False,
        **kwargs,
    ):
        """
        Generate image from text prompt and latents
        """
        do_classifier_free_guidance = guidance_scale > 1.0 # enables classifier-free guidance if guidance scale > 1.0
        self.scheduler.set_timesteps(num_inference_steps)
        # initialize values of alphas, based on the number of inference steps, so we separate 1k training steps based on num_inference_steps
        # 1k, if we have 50 as inference steps, we will have [980, 960, ..., 0] for example
        timesteps_tensor = self.scheduler.timesteps.to(self.device)
        latents = latents * self.scheduler.init_noise_sigma
        # scales the input latents to the same noise level on which the model was trained on
        # -> because VAE scales to a distribution, but might not be the same used in training, so we take it from the scheduler to be sure

        if old_text_embeddings is not None and new_text_embeddings is not None: # a bool to be sure to use the modified prompt from a certain inference step
            prompt_to_prompt = True
        else:
            prompt_to_prompt = False


        for i, t in enumerate(self.progress_bar(timesteps_tensor if not reverse_process else reversed(timesteps_tensor))):
        # this is to show a progress bar when we do inversion process
            if prompt_to_prompt:
                if i < use_old_emb_i: # here it replaces the old one with the modified one
                    text_embeddings = old_text_embeddings
                else:
                    text_embeddings = new_text_embeddings

            latent_model_input = (
                torch.cat([latents] * 2) if do_classifier_free_guidance else latents # if we're using cfg, it duplicates the latents
            )
            latent_model_input = self.scheduler.scale_model_input(latent_model_input, t)
            # So it scales the obtained latent based on the timestep. We must match the scale of the latents used during training, otherwise we would obtain garbage predictions
            noise_pred = self.unet(
                latent_model_input, t, encoder_hidden_states=text_embeddings #predicting the noise guided by text embedding
            ).sample

            if do_classifier_free_guidance:
                noise_pred_uncond, noise_pred_text = noise_pred.chunk(2) # it splits the noise prediction into two parts, one for uncod and the other cond
                noise_pred = noise_pred_uncond + guidance_scale * (
                    noise_pred_text - noise_pred_uncond
                )

            prev_timestep = ( # calculate the previous timestep index for DDIM update
                t
                - self.scheduler.config.num_train_timesteps
                // self.scheduler.num_inference_steps
            )
            # the ratio gives the interval between steps in order to retrieve the previous timestep as a subtraction with the current t.
            # num_train_timeteps is the number of diffusion steps used during training of the diffusion model (typically 1k), that defines how gradually noise is added in forward process.
            # During inference you try to do 1k in 50 inference_steps instead. So those 50 steps are spaced within the 1k-step training schedule

            if callback is not None and i % callback_steps == 0:
                callback(i, t, latents)

            # Here we compute alpha_t and alpha_t-1 (this latter one based on prev_timestep)
            alpha_prod_t = self.scheduler.alphas_cumprod[t] # this is the overline_alpha_t, the cumulative product of alphas up to time t
            alpha_prod_t_prev = ( # we calculate overline_alpha_t-1, setting it to the previous timestep (an approximation of what it was)
                self.scheduler.alphas_cumprod[prev_timestep]
                if prev_timestep >= 0
                else self.scheduler.final_alpha_cumprod # or directly to the last one
            )
            if reverse_process: # if we're doing the reverse process we will swap the directions, inverting variable values
                alpha_prod_t, alpha_prod_t_prev = alpha_prod_t_prev, alpha_prod_t
            latents = backward_ddim( # perform reverse/reconstruction process
                x_t=latents,
                alpha_t=alpha_prod_t,
                alpha_tm1=alpha_prod_t_prev,
                eps_xt=noise_pred,
            )
        return latents


    @torch.inference_mode()
    def decode_image(self, latents: torch.FloatTensor, **kwargs) -> List[Image.Image]:
        scaled_latents = 1 / 0.18215 * latents
        image = [
            self.vae.decode(scaled_latents[i : i + 1]).sample for i in range(len(latents))
        ]
        image = torch.cat(image, dim=0)
        return image

    @torch.inference_mode()
    def torch_to_numpy(self, image) -> List[Image.Image]: # this function was missing and giving error
        image = (image / 2 + 0.5).clamp(0, 1)
        image = image.cpu().permute(0, 2, 3, 1).numpy()
        return image

In [ ]:
pipe = StableDiffusionPipeline.from_pretrained("CompVis/stable-diffusion-v1-4")
#pipe.scheduler = DDIMScheduler.from_config(path / "scheduler")
# They were importing locally the scheduler. However it is a zero-shot approach so this should not be something that they have trained and that lead to some perfomance changes.
# The pipe already include its scheduler, so there's not point on importing it.
pipe = pipe.to("cuda")

model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

safety_checker/model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

...

unet/diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/806 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

# Image folder creation

Change output_folder to change the path on where the reconstructed image will go.

Change images_path to change the path where it is taking the imgs to be tested.

In [ ]:
output_folder = Path("/content/ZeroFake-Mod/reconstructed")

In [ ]:
images_path = Path("/content/ZeroFake-Mod/imgs")

### Download Chameleon and use only n of them

This following code is used to download Chameleon dataset from external Google Drive link.

In [ ]:
!pip install -q gdown

In [ ]:
import zipfile
import random

In [ ]:
!gdown 1QLYJMhy0CbBVT01BLkkw7KPPL5BpmxnH --output Chameleon.zip

In [ ]:
zip_path = "Chameleon.zip"
extract_to = images_path
n = 2

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    # Filter only files in 1_fake or 0_real
    real_files = [f for f in zip_ref.namelist() if "Chameleon/test/0_real/" in f and not f.endswith("/")]

    selected_files = random.sample(real_files, n)

    for file in selected_files:
        zip_ref.extract(file, extract_to)

In [ ]:
real_images_path = images_path / "Chameleon" / "test" / "0_real"
#real_images = natsorted(real_images_path.glob("*.*"))[:20] # to try putting selected_files here instead of [:20] here
real_images = natsorted([images_path / Path(f) for f in selected_files])

In [ ]:
images_list_str= [str(x) for x in real_images]

# Loading BLIP model to generate image prompt

In [ ]:
index = 0
nlp = spacy.load("en_core_web_sm")
model_url = 'https://storage.googleapis.com/sfr-vision-language-research/BLIP/models/model_base_capfilt_large.pth'
image_size = 512
print("Loading BLIP model...")
blipmodel = blip_decoder(pretrained=model_url, image_size=image_size, vit='base') # Error solved when using generate with this, that can be found in blip.py and med.py
print("BLIP model loaded.")
blipmodel.eval()
blipmodel = blipmodel.cuda()

print("Found", len(images_list_str), "images.")

### Most frequent words in english

In [ ]:
word_path = Path("/content/ZeroFake-Mod/words/most-common-nouns-english.csv")

In [ ]:
import pandas as pd
df = pd.read_csv(word_path)
df.head()

Here i'm filtering the list to remove articles, words that ends with punctuations and words smaller than 3 characters.

In [ ]:
df = df['Word'].str.lower()
df = df[df.str.len() > 2] #filter out elements that have len less than 2, because it was also containing articles
df = df[~df.str.contains(r'[^\w\s]', regex=True)] #filter out elements that contain punctuation

In [ ]:
import nltk
nltk.download('punkt_tab')
from nltk.corpus import wordnet as wn
download('wordnet')
download('omw-1.4')

I notice that the list was contaning many not concrete nouns and I doubted their power on the adversary prompt. For this reason I decided to try to keep only concrete nouns. Using WordNet to discriminate "physical entities" it keeps only "entities that has physical existence".


In [ ]:
def is_concrete_noun(word):
    synsets = wn.synsets(word, pos=wn.NOUN)
    for syn in synsets:
        for path in syn.hypernym_paths():
            if any(h.name().startswith("physical_entity") for h in path):
                return True
    return False

In [ ]:
df = df[df.apply(is_concrete_noun)]

In [ ]:
len(df)

In [ ]:
noun_list = [noun for noun in df]

### Brown corpora

In [ ]:
nltk.download('brown')
nltk.download('punkt_tab')

words = [word.lower() for word in brown.words()]
seen = set()
unique_words = []
for word in words:
    if word not in seen:
        seen.add(word)
        unique_words.append(word)
    if len(unique_words) >= 1000:
        break
# unique_words contains 1k words with no repetitions

text = " ".join(unique_words)
doc = nlp(text)
noun_list = [token.text for token in doc if token.pos_ == "NOUN"] #using spacy pos tagging to take only nouns

### Cosine sim & obtain perturbed prompts

In [ ]:
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [ ]:
def get_perturbed_prompts(sentence): return [replace_first_noun(sentence,noun) for noun in noun_list]

### all-MiniLM-L6-v2 embeddings

https://www.sbert.net/docs/sentence_transformer/usage/semantic_textual_similarity.html

Related papers:
- https://arxiv.org/pdf/1908.10084

In [ ]:
from sentence_transformers import SentenceTransformer, util

sentence_transformer = SentenceTransformer('all-MiniLM-L6-v2')

### FastText embeddings

In [ ]:
! wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.bin.gz
! gunzip cc.en.300.bin.gz

In [ ]:
import fasttext
ft = fasttext.load_model("cc.en.300.bin")

In [ ]:
def sentence_embedding(sentence):
    words = word_tokenize(sentence.lower())
    vectors = [ft[word] for word in words if word in ft]
    if not vectors:
        return np.zeros(ft.get_dimension())
    return np.mean(vectors, axis=0)

# DDIM Inversion and reconstruction

This code chunk represent the core of the approach, it is calling the manually defined functions of inversion and reconstruction. In fact it is not calling an instantiation of the pipeline, but using methods that we have defined such as forward diffusion and backward diffusion. Then we manually reconstruct from latents the image, that will be later used to understand the results.

In [ ]:
for impath in images_list_str:

    img = load_img(impath).unsqueeze(0).to("cuda")
    print(index)

    prompt = blipmodel.generate(img, sample=True, num_beams=3, max_length=40, min_length=5)[0]

    print(prompt)

    candidate_prompts = get_perturbed_prompts(prompt)

    #original_prompt = sentence_embedding(prompt) # to be changed with sentence_transformer.encode(prompt)
    original_prompt = sentence_transformer.encode(prompt)
    min_sim = 1.0
    selected = None
    for adv in candidate_prompts:
      #adv_prompt_word_vector = sentence_embedding(adv) # to be changed with sentence_transformer.encode(adv)
      adv_prompt_word_vector = sentence_transformer.encode(adv)
      #res = cosine_similarity(adv_prompt_word_vector,original_prompt) # to be changed with sentence_transformer.similarities(adv_prompt_word_vector,original_prompt)
      res = util.cos_sim(adv_prompt_word_vector,original_prompt)
      if res < min_sim:
        min_sim = res
        selected = adv
      #print(f"Candidate prompt: {adv} with cosine similarity score {res}")
    prompt = selected

    #prompt = replace_first_noun(prompt, target_word)

    print(prompt)

    text_embeddings = pipe.get_text_embedding(prompt)

    rng_generator=torch.Generator(device=pipe.device).manual_seed(0)

    image_latents = pipe.get_image_latents(img, rng_generator)

    reversed_latents = pipe.forward_diffusion(
        latents=image_latents,
        text_embeddings=text_embeddings,
        guidance_scale=1,
        num_inference_steps=999,
    )

    reconstructed_latents = pipe.backward_diffusion(
        latents=reversed_latents,
        text_embeddings=text_embeddings,
        guidance_scale=1,
        num_inference_steps=20,
    )

    # guidance_scale=1 so we follow the prompt only

    def latents_to_imgs(latents):
        x = pipe.decode_image(latents)
        x = pipe.torch_to_numpy(x)
        x = numpy_to_pil(x)
        return x

    image = latents_to_imgs(reconstructed_latents)[0]

    print(f"Saving image {index} to {os.path.join(output_folder, str(index) + '.png')}")
    image.save(os.path.join(output_folder, str(index) + '.png'), format="PNG")
    index += 1

# Analyzing the results

## Fréchet distance implementation

In [29]:
!pip install frechetdist

In [30]:
from frechetdist import frdist

In [47]:
def frechet_distance(gray1,gray2):
  contours1, _ = cv2.findContours(gray1, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
  #each contour is itself a NumPy array of points -> tuple = array of numpy arrays of points.
  contours2, _ = cv2.findContours(gray2, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

  #print(contours1)
  #print(contours1[0].shape) #it shows how many points in that contour

  largest_contour1 = max(contours1, key=cv2.contourArea).squeeze() #it is taking the largest one to simplify things, otherwise taking everyone could be noisy
  largest_contour2 = max(contours2, key=cv2.contourArea).squeeze()

  print(largest_contour1.shape) #(26,2)
  print(largest_contour2.shape) #(46,2)
  # PROBLEM!

  contours1_list = [tuple(pt) for pt in largest_contour1]
  contours2_list = [tuple(pt) for pt in largest_contour2]

  return frdist(contours1_list, contours2_list)

## Fréchet Inception distance implementation

The Frechet Inception Distance, or FID for short, is a metric for evaluating the quality of generated images and specifically developed to evaluate the performance of generative adversarial networks.

The FID score uses the Inception v3 model. Specifically, the coding layer of the model (the last pooling layer prior to the output classification of images) is used to capture computer-vision-specific features of an input image.

Instead of comparing raw pixels or contours, FID compares the statistics (mean and covariance) of image features extracted by Inception v3.

In [ ]:
from torchvision.models.inception import inception_v3
from scipy.linalg import sqrtm

In [ ]:
inception_model = inception_v3(pretrained=True, transform_input=False).eval()

In [ ]:
def extract_features(img):
    img = img.unsqueeze(0)

    with torch.no_grad():
        features = inception_model(img).cpu().numpy()

    return features.squeeze().cpu().numpy()

In [ ]:
def calculate_fid(features1, features2):
    mu1, mu2 = np.mean(features1, axis=0), np.mean(features2, axis=0)
    sigma1, sigma2 = np.cov(features1, rowvar=False), np.cov(features2, rowvar=False)
    covmean = sqrtm(sigma1 @ sigma2)
    fid = np.sum((mu1 - mu2) ** 2) + np.trace(sigma1 + sigma2 - 2 * covmean)
    return np.real(fid)

## Comparison cycle

In [48]:
import gc

folder1_path = real_images_path

folder2_path = output_folder

output_file = "testfake.txt"

mean_ssim_score = []

with open(output_file, "w") as f:
    f.write("Image Filename\tPixel Similarity\n")
    index = 0
    for image_path1, image_path2 in zip(natsorted(Path(folder1_path).glob("*?.*")), natsorted(Path(folder2_path).glob("*?.*"))):

      print(image_path1)
      print(image_path2)
      image1 = cv2.imread(str(image_path1))
      image2 = cv2.imread(str(image_path2))

      image1 = cv2.resize(image1, (512, 512))
      image2 = cv2.resize(image2, (512, 512))

      gray1 = cv2.cvtColor(image1, cv2.COLOR_BGR2GRAY)
      gray2 = cv2.cvtColor(image2, cv2.COLOR_BGR2GRAY)

      #frechet_distance_score = frechet_distance(gray1,gray2)
      #print(frechet_distance_score)

      features1 = extract_features(image1)
      features2 = extract_features(image2)
      fid_score = calculate_fid(features1,features2)
      print(fid_score)

      ssim_score = ssim(gray1, gray2)

      f.write(f"SSIM score of {index}: \t{ssim_score}\n")
      print("SSIM score:", ssim_score,"\n")
      index+=1
      mean_ssim_score.append(ssim_score)

      del image1, image2, gray1, gray2
      gc.collect()

    f.write(f"Mean SSIM score \t{np.mean(mean_ssim_score)}\n")
    print(f"Mean SSIM score \t{np.mean(mean_ssim_score)}\n")

print("Results saved to", output_file)

/content/ZeroFake-Mod/imgs/Chameleon/test/0_real/3eb8d6d1-c700-4026-8a0e-53469325a118.jpg
/content/ZeroFake-Mod/reconstructed/0.png
(4, 2)
(4, 2)
0.0
SSIM score: 0.6338113737120598 

/content/ZeroFake-Mod/imgs/Chameleon/test/0_real/1113c718-93ec-4291-8260-b08d41fb855d.jpg
/content/ZeroFake-Mod/reconstructed/1.png
(26, 2)
(46, 2)


ValueError: Input curves do not have the same dimensions.

In case you want to download the folders containing images.

In [ ]:
from google.colab import files
import shutil

shutil.make_archive("reconstructed", 'zip', "/content/ZeroFake-Mod/reconstructed")
files.download("reconstructed.zip")

shutil.make_archive("imgs", 'zip', "/content/ZeroFake-Mod/imgs")
files.download("imgs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Other code

In order to retrieve the exact correspondent real image from COCO2017, the following code was used.

In [ ]:
import urllib.request
import ssl

url = "https://images.cocodataset.org/train2017/000000301684.jpg"

ssl._create_default_https_context = ssl._create_unverified_context

try:
    urllib.request.urlretrieve(url, "000000301684.jpg")
    print("Image downloaded successfully.")
except urllib.error.URLError as e:
    print(f"Error downloading image: {e}")

from PIL import Image
try:
    img = Image.open("000000301684.jpg")
    img.show()
except FileNotFoundError:
    print("Downloaded image file not found.")

Since the real image taken from RAISE is in .NEF extension (that’s NIKON camera extension), the image was converted into a standard format like PNG, otherwise the model performs really bad giving weird results.

In [ ]:
!pip install rawpy

In [ ]:
import rawpy
from PIL import Image
import numpy as np

with rawpy.imread('/content/r0141f0c9t.NEF') as raw:
    rgb = raw.postprocess()

img = Image.fromarray(rgb)
img.save('r0141f0c9t_mod.png', format='PNG')

### Download Chameleon and use only n of them

This following code is used to download Chameleon dataset from external Google Drive link.

In [ ]:
!pip install -q gdown

In [ ]:
import zipfile

In [ ]:
!gdown 1QLYJMhy0CbBVT01BLkkw7KPPL5BpmxnH --output Chameleon.zip

In [ ]:
zip_path = "Chameleon.zip"
extract_to = "/content/data"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    # Filter only files in 0_real
    real_files = [f for f in zip_ref.namelist() if "Chameleon/test/0_real/" in f and not f.endswith("/")]

    for file in real_files[:20]:
        zip_ref.extract(file, extract_to)

In [ ]:
real_images_path = Path("/content/data/Chameleon/test/0_real")
real_images = natsorted(real_images_path.glob("*.*"))[:20]

In [ ]:
import matplotlib.pyplot as plt
img = cv2.imread(str(real_images[15]))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()